# Trustworthy GeoAI — Course Work Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vezarachan/TUM_Course_SVA_TrustGeoAI/blob/main/coursework/TrustGeoAI_CourseWork.ipynb)

**Spatial Visual Analytics · Trustworthy GeoAI**

> **Easiest way to run this:** click the **Open in Colab** badge above — you only need a Google
> account, nothing to install. The first setup cell downloads everything it needs automatically.

You will investigate **where a GeoAI model can and cannot be trusted**, using *spatial uncertainty*
(prediction intervals from conformal prediction) as the trust signal.

### How to use this notebook  ▶️
1. Run everything once: menu **Run ▸ Run All Cells** (or `Shift+Enter` down the notebook).
2. Then use the **drop-down menus** to choose a dataset, a method, and — most importantly —
   **different visualization methods**.
3. Your job is to **think and compare**: *which visualization best reveals the trustworthiness of
   the model, and what does it tell you?* Write your answers in the 🟨 **YOUR TURN** cells.

> You barely need to write code. The computation is done for you — spend your effort on **choosing
> visualizations**, optionally **building your own view in a few lines**, and **interpreting trust**.
> Plugging in your own idea is how your **unique perspective** comes through (see section ②③).
> The four aspects we grade:
>
> 1. **The scientific question** · 2. **The design of user interface** ·
> 3. **The way of revealing trustworthiness** · 4. **Retrospect the limits of AI tools**

---

## Setup  🟦 PROVIDED — just run it

Imports, the `geocp` method, metadata helpers, the analysis engine, and the **visualization library**.

In [ ]:
import sys, os, re, json, math, io, base64, inspect, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")

# ---- Where to get data/code: works on Google Colab, locally, or a fresh clone ----
REPO, BRANCH = "Vezarachan/TUM_Course_SVA_TrustGeoAI", "main"
RAW = "https://raw.githubusercontent.com/" + REPO + "/" + BRANCH
IN_COLAB = "google.colab" in sys.modules

def _fetch(repo_path, local_path):
    """Download a file from the course GitHub repo if it is not already present."""
    if not os.path.exists(local_path):
        d = os.path.dirname(local_path)
        if d:
            os.makedirs(d, exist_ok=True)
        urllib.request.urlretrieve(RAW + "/" + repo_path, local_path)
    return local_path

# Make geocp importable; download it from GitHub if missing (e.g. on Colab).
HERE = os.getcwd()
ROOT = os.path.dirname(HERE) if os.path.basename(HERE) in ("coursework", "live_demo") else HERE
for p in [HERE, ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
try:
    from geocp import GeoConformalRegressor
except ModuleNotFoundError:
    for _f in ["__init__.py", "core.py", "estimators.py", "results.py", "utils.py", "weights.py"]:
        _fetch("geocp/" + _f, "geocp/" + _f)
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    from geocp import GeoConformalRegressor

from sklearn.ensemble import RandomForestRegressor   # method: "geocp" or "bayesian"

# Use a local dataset folder if present; otherwise files download on demand.
DATA_DIR = next((d for d in [os.path.join(HERE, "dataset"), os.path.join(ROOT, "dataset")]
                 if os.path.isdir(d)), "dataset")
META_PATH = os.path.join(DATA_DIR, "Metadata.xlsx")

# Datasets you can choose from (each is downloaded the first time it is used).
CSV_FILES = ["US_Health_DIABETES.csv", "US_Health_OBESITY.csv", "US_Health_STROKE.csv",
             "US_Health_CANCER.csv", "US_Health_ARTHRITIS.csv", "US_Health_BPHIGH.csv",
             "US_Health_CASTHMA.csv", "US_Health_DEPRESSION.csv", "US_Hydro_CAMELS.csv",
             "US_Politics_Voting.csv", "US_Forest_FIA.csv", "US_Climate_ERA5_CLIMATE.csv"]

def ensure_csv(name):
    """Local path to a dataset CSV, downloading it from GitHub if needed."""
    return _fetch("coursework/dataset/" + name, os.path.join(DATA_DIR, name))

# Pull the small metadata dictionary so column descriptions work everywhere.
try:
    _fetch("coursework/dataset/metadata.json", os.path.join(DATA_DIR, "metadata.json"))
except Exception:
    pass

# ---- metadata helpers (one sheet per dataset in Metadata.xlsx) ----
def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower().replace(".csv", ""))

_META_JSON = None
_META_WARNED = False
def _load_meta_json():
    global _META_JSON
    if _META_JSON is None:
        p = os.path.join(DATA_DIR, "metadata.json")
        try:
            with open(p, encoding="utf-8") as f:
                _META_JSON = json.load(f)
        except Exception:
            _META_JSON = {}
    return _META_JSON

def _pick_sheet(names, key):
    return ([s for s in names if _norm(s) == key]
            or [s for s in names if key in _norm(s) or _norm(s) in key])

def meta_for(name):
    """Column dictionary for a dataset. Prefers metadata.json (no dependency);
    falls back to Metadata.xlsx (needs openpyxl)."""
    global _META_WARNED
    key = _norm(name)
    mj = _load_meta_json()                      # 1) preferred: bundled JSON
    if mj:
        m = _pick_sheet(list(mj), key)
        if m:
            sh = mj[m[0]]
            return pd.DataFrame(sh["rows"], columns=sh["columns"])
    try:                                        # 2) fallback: read the .xlsx
        xl = pd.ExcelFile(META_PATH)
    except Exception:
        if not mj and not _META_WARNED:
            print("Note: no metadata.json found and openpyxl not installed - column")
            print("      dictionaries are skipped (everything else still works).")
            _META_WARNED = True
        return None
    m = _pick_sheet(xl.sheet_names, key)
    if not m:
        return None
    return xl.parse(m[0]).dropna(how="all", axis=1).dropna(how="all", axis=0)

def dataset_catalog():
    rows = []
    for f in CSV_FILES:
        m = meta_for(f); target = ""
        if m is not None and "Full Name" in m.columns:
            yr = m[m.iloc[:, 0].astype(str).str.strip() == "Y"]
            if len(yr):
                target = str(yr["Full Name"].iloc[0])
        rows.append({"dataset": f, "predicts (Y)": target})
    return pd.DataFrame(rows)

def dataset_info(name):
    df = pd.read_csv(ensure_csv(name))
    print(f"{name}   —   {df.shape[0]} rows x {df.shape[1]} columns")
    m = meta_for(name)
    if m is not None:
        print("\nColumn dictionary (what Y and each X mean):")
        display(m)
    else:
        print("\n(no metadata sheet) columns:", list(df.columns))
    print("\nPreview:"); display(df.head())
    print("\nNumeric summary:"); display(df.describe().round(2))
    return df

print("Setup complete.")

### The analysis engine &amp; visualization library  🟦 PROVIDED — just run it

`run_pipeline()` trains a model and runs the chosen method (results are cached).
`VIZ` is a dictionary of ready-made **visualization methods** — you pick from these.

In [ ]:
# ===== ANALYSIS ENGINE (cached) =====================================
_PIPE = {}

def _numeric_features(df, feat):
    """Make every feature numeric: parse numeric strings, factorize categoricals."""
    cols = []
    for c in feat:
        s = df[c]
        if s.dtype == object:
            sn = pd.to_numeric(s, errors="coerce")
            if sn.notna().mean() > 0.5:            # mostly numeric -> use as number
                cols.append(sn.rename(c))
            else:                                   # categorical text -> integer codes
                cols.append(pd.Series(pd.factorize(s)[0], index=s.index, name=c).astype(float))
        else:
            cols.append(s.astype(float))
    return pd.concat(cols, axis=1)

def run_pipeline(dataset, method="geocp", alpha=0.10, bandwidth=0.3):
    key = (dataset, method, alpha, bandwidth)
    if key in _PIPE:
        return _PIPE[key]
    df = pd.read_csv(ensure_csv(dataset))
    if "Y" not in df.columns:
        raise ValueError(dataset + " has no 'Y' column - pick a dataset listed by the catalog.")
    feat = [c for c in df.columns if c.startswith("X")]
    if {"lon", "lat"}.issubset(df.columns):
        cc = ["lon", "lat"]
    elif {"proj_x", "proj_y"}.issubset(df.columns):
        cc = ["proj_x", "proj_y"]
    elif {"proj_X", "proj_Y"}.issubset(df.columns):
        cc = ["proj_X", "proj_Y"]
    else:
        raise ValueError("No lon/lat or proj_x/proj_y coordinates in " + dataset)

    # Build numeric frames, then drop any row with missing feature/target/coordinate.
    Xdf = _numeric_features(df, feat)
    ydf = pd.to_numeric(df["Y"], errors="coerce")
    cdf = df[cc].apply(pd.to_numeric, errors="coerce")
    keep = Xdf.notna().all(axis=1) & ydf.notna() & cdf.notna().all(axis=1)
    df, Xdf, ydf, cdf = (df[keep].reset_index(drop=True), Xdf[keep].reset_index(drop=True),
                         ydf[keep].reset_index(drop=True), cdf[keep].reset_index(drop=True))
    if len(df) < 50:
        raise ValueError(dataset + " has too few clean rows after dropping missing values.")
    if len(df) > 1800:                              # keep it fast
        samp = df.sample(1800, random_state=0).index
        df, Xdf, ydf, cdf = (df.loc[samp].reset_index(drop=True), Xdf.loc[samp].reset_index(drop=True),
                             ydf.loc[samp].reset_index(drop=True), cdf.loc[samp].reset_index(drop=True))

    X = Xdf.to_numpy(float); y = ydf.to_numpy(float); coords = cdf.to_numpy(float)
    cstd = (coords - coords.mean(0)) / (coords.std(0) + 1e-9)   # +eps guards constant coords
    rng = np.random.default_rng(0); idx = rng.permutation(len(df))
    a, b = int(0.6 * len(df)), int(0.8 * len(df))
    fi, ci, ti = idx[:a], idx[a:b], idx[b:]
    model = RandomForestRegressor(n_estimators=250, random_state=0, n_jobs=-1).fit(X[fi], y[fi])
    reg = GeoConformalRegressor(model.predict, X[ci], y[ci], cstd[ci],
                                bandwidth=bandwidth, miscoverage_level=alpha)
    res = reg.geo_conformalize(X[ti], y[ti], cstd[ti], bayesian=(method == "bayesian"))
    R = pd.DataFrame({"lon": coords[ti, 0], "lat": coords[ti, 1],
                      "pred": res.pred_value, "truth": res.true_value,
                      "lower": res.lower_bound, "upper": res.upper_bound,
                      "uncertainty": res.uncertainty})
    R["width"]   = R.upper - R.lower
    R["error"]   = (R.pred - R.truth).abs()
    R["covered"] = (R.truth >= R.lower) & (R.truth <= R.upper)
    if res.is_bayesian:
        R["posterior_std"] = res.posterior_std
    if "region" in df.columns:
        R["region"] = df["region"].values[ti]
    R.attrs.update(alpha=alpha, coverage=res.coverage, method=method, dataset=dataset)
    summary = (f"{dataset}  |  method={method}  |  coverage {res.coverage:.1%} "
               f"(target {1-alpha:.0%})  |  mean width {res.mean_width_finite:.3f}")
    _PIPE[key] = (R, summary)
    return _PIPE[key]

# ===== STYLE — colors / basemap / point size (set by the explorer controls) =====
STYLE = {"cmap": "auto", "basemap": True, "size": 18}
PALETTES = ["auto", "viridis", "plasma", "magma", "cividis", "coolwarm",
            "YlOrRd", "YlGnBu", "Spectral", "RdYlGn", "Blues", "Reds"]

def _eff_cmap(warn):
    c = STYLE.get("cmap", "auto")
    return c if c and c != "auto" else ("YlOrRd" if warn else "viridis")

# ---- US-states basemap (drawn only when coordinates look like lon/lat) ----
_STATES = None
def _load_states():
    global _STATES
    if _STATES is None:
        try:
            p = _fetch("coursework/assets/us_states.geojson", os.path.join(DATA_DIR, "us_states.geojson"))
            with open(p, encoding="utf-8") as f:
                _STATES = json.load(f)
        except Exception:
            _STATES = {"features": []}
    return _STATES

def _is_geographic(R):
    return bool(R["lon"].between(-180, 180).all() and R["lat"].between(-90, 90).all())

def _basemap(ax, R):
    if not STYLE.get("basemap", True) or not _is_geographic(R):
        return
    for feat in _load_states().get("features", []):
        geom = feat.get("geometry") or {}
        t = geom.get("type"); coords = geom.get("coordinates") or []
        if t == "Polygon":
            rings = coords
        elif t == "MultiPolygon":
            rings = [r for part in coords for r in part]
        else:
            rings = []
        for ring in rings:
            xs = [c[0] for c in ring]; ys = [c[1] for c in ring]
            ax.fill(xs, ys, color="#f3f4f6", zorder=0)
            ax.plot(xs, ys, color="#c9ccd1", lw=0.5, zorder=1)

def _newax(ax, w=7, h=5):
    if ax is not None:
        return ax, False
    fig, ax = plt.subplots(figsize=(w, h))
    return ax, True

def _scatter_map(ax, R, values, label, warn):
    v = np.asarray(values, float); m = np.isfinite(v)
    _basemap(ax, R)
    sc = ax.scatter(R["lon"][m], R["lat"][m], c=v[m], cmap=_eff_cmap(warn),
                    s=STYLE["size"], edgecolor="white", linewidth=.2, zorder=3)
    ax.figure.colorbar(sc, ax=ax, shrink=.8, label=label)
    ax.set_xlabel("lon"); ax.set_ylabel("lat")

# ===== VISUALIZATION LIBRARY (each accepts an optional `ax` for dashboards) =====
def _fin(R, col):
    return R[np.isfinite(R[col])]

def v_unc(R, ax=None):
    ax, own = _newax(ax); _scatter_map(ax, R, R["uncertainty"], "interval half-width", True)
    ax.set_title("Uncertainty — where is the model unsure?")
    if own: plt.tight_layout(); plt.show()

def v_pred(R, ax=None):
    ax, own = _newax(ax); _scatter_map(ax, R, R["pred"], "predicted value", False)
    ax.set_title("Prediction — the predicted values")
    if own: plt.tight_layout(); plt.show()

def v_err(R, ax=None):
    ax, own = _newax(ax); _scatter_map(ax, R, R["error"], "|prediction - truth|", True)
    ax.set_title("Error — where is the model actually wrong?")
    if own: plt.tight_layout(); plt.show()

def v_cover_map(R, ax=None):
    ax, own = _newax(ax); _basemap(ax, R)
    ax.scatter(R["lon"][R.covered],  R["lat"][R.covered],  s=max(6, STYLE["size"]-3), c="#cfcfcf", label="covered", zorder=3)
    ax.scatter(R["lon"][~R.covered], R["lat"][~R.covered], s=STYLE["size"]+2,        c="#dc2626", label="missed", zorder=4)
    ax.legend(loc="best"); ax.set_title("Coverage hit / miss — where do intervals fail?")
    ax.set_xlabel("lon"); ax.set_ylabel("lat")
    if own: plt.tight_layout(); plt.show()

def v_width_hist(R, ax=None):
    ax, own = _newax(ax, 7, 4.2); d = _fin(R, "width")
    ax.hist(d.width, bins=30, color="#2563eb", alpha=.8)
    ax.axvline(d.width.mean(), color="black", ls="--", label=f"mean {d.width.mean():.2f}")
    ax.set_xlabel("prediction-interval width"); ax.set_ylabel("count")
    ax.set_title("Interval-width histogram"); ax.legend()
    if own: plt.tight_layout(); plt.show()

def v_honesty(R, ax=None):
    ax, own = _newax(ax, 6, 5); d = _fin(R, "uncertainty")
    ax.scatter(d.uncertainty, d.error, s=12, alpha=.5)
    lim = max(d.uncertainty.max(), d.error.max())
    ax.plot([0, lim], [0, lim], "--", color="grey", label="y = x")
    ax.set_xlabel("predicted uncertainty"); ax.set_ylabel("actual |error|")
    ax.set_title("Honesty check — claimed vs real error"); ax.legend()
    if own: plt.tight_layout(); plt.show()

def v_reliability(R, ax=None):
    ax, own = _newax(ax, 6, 5); d = _fin(R, "uncertainty").sort_values("uncertainty")
    if len(d) < 10:
        ax.text(.5, .5, "too few points", ha="center")
    else:
        groups = np.array_split(d, 10)
        mu = [g.uncertainty.mean() for g in groups]; me = [g.error.mean() for g in groups]
        ax.plot(mu, me, "o-")
        lim = max(max(mu), max(me)); ax.plot([0, lim], [0, lim], "--", color="grey", label="ideal")
        ax.set_xlabel("mean predicted uncertainty"); ax.set_ylabel("mean actual |error|"); ax.legend()
    ax.set_title("Reliability — more uncertainty should mean more error")
    if own: plt.tight_layout(); plt.show()

def v_grid(R, ax=None):
    ax, own = _newax(ax); d = _fin(R, "uncertainty"); _basemap(ax, R)
    hb = ax.hexbin(d.lon, d.lat, C=d.uncertainty, gridsize=18, cmap=_eff_cmap(True),
                   reduce_C_function=np.mean, zorder=2)
    ax.figure.colorbar(hb, ax=ax, shrink=.8, label="mean half-width")
    ax.set_title("Spatial grid — mean uncertainty per cell"); ax.set_xlabel("lon"); ax.set_ylabel("lat")
    if own: plt.tight_layout(); plt.show()

def v_region(R, ax=None):
    ax, own = _newax(ax, 7, 5)
    if "region" not in R.columns:
        ax.text(.5, .5, "no region column in this dataset", ha="center")
    else:
        g = R.groupby("region")["covered"].mean().sort_values()
        ax.barh(g.index.astype(str), g.values * 100, color=plt.cm.RdYlGn(g.values))
        tgt = (1 - R.attrs.get("alpha", .1)) * 100
        ax.axvline(tgt, color="black", ls="--", label=f"target {tgt:.0f}%"); ax.legend()
        ax.set_xlabel("coverage (%)")
    ax.set_title("Regional coverage — is the guarantee local too?")
    if own: plt.tight_layout(); plt.show()

def v_meta(R, ax=None):
    ax, own = _newax(ax)
    if "posterior_std" not in R.columns:
        ax.text(.5, .5, 'set method = "bayesian" to see this', ha="center")
        ax.set_title("Bayesian meta-uncertainty")
    else:
        _scatter_map(ax, R, R["posterior_std"], "posterior std", True)
        ax.set_title("Bayesian meta-uncertainty — how unsure is the uncertainty?")
    if own: plt.tight_layout(); plt.show()

VIZ = {
    "Uncertainty map (where is the model unsure?)":  v_unc,
    "Prediction map (the values)":                   v_pred,
    "Absolute-error map (where is it wrong?)":       v_err,
    "Coverage hit/miss map (where intervals fail)":  v_cover_map,
    "Interval-width histogram":                      v_width_hist,
    "Honesty check: uncertainty vs error":           v_honesty,
    "Reliability: binned uncertainty vs error":      v_reliability,
    "Spatial grid: mean uncertainty (hexbin)":       v_grid,
    "Regional coverage bars":                        v_region,
    "Bayesian meta-uncertainty map":                 v_meta,
}

# ---- helpers so YOUR OWN view needs only a line or two (accept `ax` for dashboards) ----
def quickmap(R, values, title="", cmap=None, ax=None):
    """Plot any per-point values on the map (basemap + your chosen palette)."""
    ax2, own = _newax(ax)
    eff = cmap or _eff_cmap(False)
    v = np.asarray(values, float); m = np.isfinite(v)
    _basemap(ax2, R)
    sc = ax2.scatter(R["lon"][m], R["lat"][m], c=v[m], cmap=eff,
                     s=STYLE["size"], edgecolor="white", linewidth=.2, zorder=3)
    ax2.figure.colorbar(sc, ax=ax2, shrink=.8)
    ax2.set_title(title); ax2.set_xlabel("lon"); ax2.set_ylabel("lat")
    if own: plt.tight_layout(); plt.show()

def quickhist(values, title="", bins=30, color="#2563eb", ax=None):
    """Plot a histogram of any values."""
    ax2, own = _newax(ax, 7, 4)
    v = np.asarray(values, float); v = v[np.isfinite(v)]
    ax2.hist(v, bins=bins, color=color, alpha=.8); ax2.set_title(title)
    if own: plt.tight_layout(); plt.show()

def trust_view(name):
    """Decorator: register YOUR function as a visualization named `name`.
    It appears in the explorer, the dashboard, and the report automatically."""
    def deco(fn):
        VIZ[name] = fn
        return fn
    return deco

def _call_view(fn, R, ax):
    if "ax" in inspect.signature(fn).parameters:
        fn(R, ax=ax)
    else:                       # custom view that makes its own figure
        ax.axis("off"); ax.text(.5, .5, "(your view drew its own figure above)", ha="center", va="center")
        fn(R)

def render_dashboard(R, views, ncols=2, show=True):
    """Render a chosen COMBINATION of views in a grid; returns the Figure."""
    views = [v for v in views if v in VIZ]
    if not views:
        print("Select at least one visualization."); return None
    ncols = max(1, min(int(ncols), len(views))); nrows = math.ceil(len(views) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.8 * nrows), squeeze=False)
    flat = [a for row in axes for a in row]
    for ax, name in zip(flat, views):
        try:
            _call_view(VIZ[name], R, ax)
        except Exception as e:
            ax.axis("off"); ax.text(.5, .5, "error: " + str(e), ha="center")
    for ax in flat[len(views):]:
        ax.axis("off")
    fig.tight_layout()
    if show: plt.show()
    return fig

def fig_to_base64(fig, dpi=120):
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    buf.seek(0); return base64.b64encode(buf.read()).decode("ascii")

def gallery(R, names=None):
    for nm in (names or list(VIZ)):
        print("\n" + "=" * 70 + "\n" + nm); VIZ[nm](R)

print("Engine ready. Visualization methods available:")
for k in VIZ: print("  -", k)

### Browse the available datasets  🟦 PROVIDED

Each dataset and the quantity it predicts. Use it to choose one for your study.

In [ ]:
dataset_catalog()

---
## ① The scientific question  🟨 YOUR TURN

Choose **one dataset** below (set `DATASET`). Run the next cell to read its **column dictionary**
(what `Y` and the `X` features mean) and a data preview. Then write your scientific question.

In [ ]:
# The only setting you must choose for aspect ①:
DATASET = "US_Health_DIABETES.csv"   # change to any name from the catalog above

In [ ]:
df_preview = dataset_info(DATASET)

**① Write your scientific question** (double-click to edit this cell).

> *TODO — e.g. Can we trust the model's predicted diabetes prevalence across US states, and in
> which regions is it least reliable? Why might trust break down there?*

Replace the line above with your own question.

---
## ② Design of the interface  &amp;  ③ Revealing trustworthiness  🟨 YOUR TURN

This is the heart of the course work. The explorer below is a small **dashboard builder** — use the
controls to compose the picture *you* want to tell:

- **dataset / method** — what you study, and how trust is quantified.
- **charts** (multi-select) — pick **several** visualizations to show **together** and compare.
- **palette** — choose the color scheme; **US basemap** on/off; **point size**; grid **columns**.
- **alpha** (target coverage) and **bandwidth** (how local the method is) — tune the story.

You have full freedom at two levels: (1) combine the ready-made charts, and (2) **build your own
view** (next cell) — it appears in the same *charts* list automatically. As you explore, ask:
*which combination makes the model's (un)trustworthiness clearest, and why?*

### Build your own visualization  🟨 YOUR TURN (optional but encouraged)

You are given a tidy table **`R`** (one row per test point) with these columns:

| column | meaning |
|---|---|
| `lon`, `lat` | location |
| `pred`, `truth` | predicted value, true value |
| `lower`, `upper`, `uncertainty` | prediction interval and its half-width |
| `width`, `error` | interval width, and actual `|pred - truth|` |
| `covered` | `True` if the truth fell inside the interval |
| `region` | region label (most datasets) |
| `posterior_std` | meta-uncertainty (only for `method = bayesian`) |

Helpers (so you write almost no code): `quickmap(R, values, title, cmap)` and
`quickhist(values, title)`. Or use `plt` directly for full control. Register a view with the
`@trust_view("name")` decorator — then it appears in the explorer menu and the gallery.

In [ ]:
# ===================================================================
# ★ YOUR TURN — build your OWN view of trust. A few lines is enough.
#   Edit the example, rename it, add more. Then RUN the explorer cell below.
# ===================================================================

@trust_view("MY VIEW — over/under-confidence (rename me)")
def my_view(R, ax=None):
    # Is the model honest? ratio = claimed uncertainty / real error.
    #   > 1 : cautious (intervals wider than needed)   < 1 : over-confident (too narrow)
    ratio = R["uncertainty"] / (R["error"] + 1e-9)
    quickmap(R, ratio, "uncertainty / error   (>1 cautious, <1 over-confident)",
             cmap="coolwarm", ax=ax)   # keep the `ax=ax` so it works inside the dashboard

# You can register as many views as you like — each becomes a chart option:
# @trust_view("My second idea")
# def my_view2(R, ax=None):
#     quickhist(R["width"], "interval widths", ax=ax)
print("Custom views registered. Now run the explorer cell below.")

In [ ]:
# === INTERACTIVE DASHBOARD EXPLORER — use the controls; no coding needed ===
# (Re-run this cell whenever you add or edit a custom view above.)
def explore(dataset="US_Health_DIABETES.csv", method="geocp", alpha=0.10, bandwidth=0.30,
            palette="auto", basemap=True, point_size=18, columns=2, charts=()):
    STYLE["cmap"] = palette; STYLE["basemap"] = basemap; STYLE["size"] = int(point_size)
    R, summary = run_pipeline(dataset, method, alpha, bandwidth)
    print(summary, "\n")
    render_dashboard(R, list(charts) or [list(VIZ)[0]], ncols=columns)

try:
    import ipywidgets as widgets
    _sl = dict(continuous_update=False)
    ui = widgets.interactive(
        explore,
        dataset=widgets.Dropdown(options=CSV_FILES, value=DATASET, description="dataset"),
        method=widgets.Dropdown(options=["geocp", "bayesian"], value="geocp", description="method"),
        alpha=widgets.FloatSlider(value=0.10, min=0.02, max=0.30, step=0.02, description="alpha", readout_format=".2f", **_sl),
        bandwidth=widgets.FloatSlider(value=0.30, min=0.10, max=1.00, step=0.05, description="bandwidth", **_sl),
        palette=widgets.Dropdown(options=PALETTES, value="auto", description="palette"),
        basemap=widgets.Checkbox(value=True, description="US basemap"),
        point_size=widgets.IntSlider(value=18, min=4, max=60, step=2, description="point size", **_sl),
        columns=widgets.IntSlider(value=2, min=1, max=3, description="columns", **_sl),
        charts=widgets.SelectMultiple(
            options=list(VIZ), rows=11, description="charts",
            value=("Uncertainty map (where is the model unsure?)",
                   "Coverage hit/miss map (where intervals fail)")),
    )
    display(ui)
except Exception as e:
    print("(ipywidgets not available — call explore(...) by hand instead)")
    print('e.g.  explore(DATASET, "geocp", charts=["Uncertainty map (where is the model unsure?)",')
    print('                                        "Coverage hit/miss map (where intervals fail)"])\n')
    explore(DATASET, "geocp", charts=["Uncertainty map (where is the model unsure?)",
                                      "Coverage hit/miss map (where intervals fail)"])

**② Write-up — interface design.** Which visualization method(s) did you choose to present trust,
and **why**? What should a viewer notice first? *(2-4 sentences.)*

> *TODO: your answer here.*

**Optional — compare several at once.** Run the cell below to see a gallery of all methods for your
current `DATASET` / method, so you can compare them side by side before deciding.

In [ ]:
R, summary = run_pipeline(DATASET, "geocp")
print(summary)
gallery(R)            # or e.g.  gallery(R, ["Uncertainty map (where is the model unsure?)",
                      #                       "Coverage hit/miss map (where intervals fail)"])

**③ Write-up — revealing trustworthiness.** Based on the visualization(s) you chose, **where can the
model be trusted and where not?** Point to a concrete region or pattern, and say what evidence in the
plot supports it. *(3-6 sentences.)*

> *TODO: your answer here.*

---
## ④ Retrospect the limits of AI tools  🟨 YOUR TURN

Run the pointers below (uncertifiable points, local vs. global coverage), then reflect.

In [ ]:
R, summary = run_pipeline(DATASET, "geocp")
frac_inf = float(np.mean(~np.isfinite(R.uncertainty)))
print(summary)
print(f"Points the method could NOT certify (infinite interval): {frac_inf:.1%}")
if "region" in R.columns:
    print("\nLocal coverage by region (a global guarantee can hide local failure):")
    print(R.groupby("region")["covered"].mean().round(2).sort_values())

**④ Write-up — limits.** Where did the method (or an LLM you used) fall short? What stays a **human
judgment** that no model or visualization can settle? *(3-6 sentences.)*

> *TODO: your answer here.*

---
## 📤 Export your report  🟨 YOUR TURN

Turn your exploration into a **standalone `report.html`** — your unique deliverable. Fill in the
fields, choose the **charts that tell your story**, pick the palette/basemap, and click
**Generate report.html**. On Colab it downloads automatically; locally it is written next to the
notebook. Open it in any browser — it embeds your figures and your four write-ups, no dependencies.

In [ ]:
import html as _html

def build_report(title, author, dataset, method, alpha, bandwidth, palette, basemap, point_size,
                 columns, charts, q, interface, findings, limits, filename="report.html"):
    """Render the chosen dashboard + narrative into a self-contained report.html."""
    STYLE["cmap"] = palette; STYLE["basemap"] = basemap; STYLE["size"] = int(point_size)
    R, summary = run_pipeline(dataset, method, alpha, bandwidth)
    fig = render_dashboard(R, list(charts) or [list(VIZ)[0]], ncols=columns, show=False)
    img = fig_to_base64(fig) if fig is not None else ""
    if fig is not None:
        plt.close(fig)
    esc = _html.escape
    css = ("<style>body{font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;"
           "max-width:900px;margin:32px auto;padding:0 18px;color:#1a1a1a;line-height:1.6}"
           "h1{margin-bottom:2px}.meta{color:#6b7280;font-size:14px}h2{font-size:18px;margin-top:26px}"
           "img{max-width:100%;border:1px solid #e3e3e3;border-radius:8px}"
           ".box{background:#f7f7f8;border:1px solid #e3e3e3;border-radius:8px;padding:10px 14px;"
           "font-size:13px;color:#444}</style>")
    setup = (f"<b>Setup:</b> dataset = {esc(str(dataset))} &middot; method = {esc(str(method))} "
             f"&middot; target coverage = {(1-alpha)*100:.0f}% &middot; bandwidth = {bandwidth} "
             f"&middot; palette = {esc(str(palette))}<br>"
             f"<span style='color:#6b7280'>{esc(summary)}</span>")
    secs = [("1. The scientific question", q), ("2. The design of user interface", interface),
            ("3. The way of revealing trustworthiness", findings),
            ("4. Retrospect the limits of AI tools", limits)]
    body = ""
    for t, v in secs:
        body += "<h2>" + esc(t) + "</h2><p>" + esc(str(v)).replace(chr(10), "<br>") + "</p>"
    doc = ("<!doctype html><html><head><meta charset='utf-8'><title>" + esc(str(title)) + "</title>"
           + css + "</head><body>"
           + "<h1>" + esc(str(title)) + "</h1>"
           + "<div class='meta'>" + esc(str(author)) + " &middot; Trustworthy GeoAI course work</div>"
           + "<div class='box'>" + setup + "</div>"
           + "<h2>Visualization</h2><img alt='dashboard' src='data:image/png;base64," + img + "'>"
           + body + "</body></html>")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(doc)
    print("Wrote", filename, "(" + str(len(doc) // 1024) + " KB).")
    try:
        from google.colab import files
        files.download(filename)
    except Exception:
        print("Open / download", filename, "from the file browser (left panel).")
    return filename

try:
    import ipywidgets as widgets
    _L = widgets.Layout(width="98%")
    W = {
        "title":     widgets.Text(value="My Trust Report", description="title", layout=_L),
        "author":    widgets.Text(value="", description="your name", layout=_L),
        "dataset":   widgets.Dropdown(options=CSV_FILES, value=DATASET, description="dataset"),
        "method":    widgets.Dropdown(options=["geocp", "bayesian"], description="method"),
        "alpha":     widgets.FloatSlider(value=0.10, min=0.02, max=0.30, step=0.02, description="alpha", readout_format=".2f"),
        "bandwidth": widgets.FloatSlider(value=0.30, min=0.10, max=1.00, step=0.05, description="bandwidth"),
        "palette":   widgets.Dropdown(options=PALETTES, value="auto", description="palette"),
        "basemap":   widgets.Checkbox(value=True, description="US basemap"),
        "columns":   widgets.IntSlider(value=2, min=1, max=3, description="columns"),
        "charts":    widgets.SelectMultiple(options=list(VIZ), rows=8, description="charts",
                        value=("Uncertainty map (where is the model unsure?)",
                               "Coverage hit/miss map (where intervals fail)")),
        "q":         widgets.Textarea(description="① question",  layout=widgets.Layout(width="98%", height="60px")),
        "interface": widgets.Textarea(description="② interface", layout=widgets.Layout(width="98%", height="60px")),
        "findings":  widgets.Textarea(description="③ findings",  layout=widgets.Layout(width="98%", height="80px")),
        "limits":    widgets.Textarea(description="④ limits",    layout=widgets.Layout(width="98%", height="80px")),
    }
    _btn = widgets.Button(description="Generate report.html", button_style="primary", icon="download")
    _out = widgets.Output()
    def _go(_):
        with _out:
            _out.clear_output()
            build_report(W["title"].value, W["author"].value, W["dataset"].value, W["method"].value,
                         W["alpha"].value, W["bandwidth"].value, W["palette"].value, W["basemap"].value,
                         18, W["columns"].value, W["charts"].value,
                         W["q"].value, W["interface"].value, W["findings"].value, W["limits"].value)
    _btn.on_click(_go)
    display(widgets.VBox(list(W.values()) + [_btn, _out]))
except Exception as e:
    print("ipywidgets not available — call build_report(...) directly, e.g.:")
    print('build_report("My Report", "Your Name", "US_Health_DIABETES.csv", "geocp", 0.10, 0.30,')
    print('   "auto", True, 18, 2,')
    print('   ["Uncertainty map (where is the model unsure?)", "Coverage hit/miss map (where intervals fail)"],')
    print('   "my question", "my interface rationale", "my trust findings", "my reflection on limits")')

---
## Submission checklist

For each aspect, make sure your notebook contains:

- **①** your dataset choice + a clear scientific question
- **②** the visualization method(s) you chose + why (interface design)
- **③** an interpretation of where the model can/cannot be trusted, citing the plot
- **④** a reflection on the limits of the method and of AI tools
- **export** your `report.html` (the *Export your report* section) and submit it together with the notebook

*Tip:* switching `method` to `bayesian`, changing the **palette/basemap**, or comparing two datasets
often changes the trust picture — great material for ③ and ④.